# 🛒 Retail Dataset Analysis 02 - Customer Segmentation & Pricing

Following a previous analysis on performence and product mix, this notebook performs a deep-dive exploratory analysis into retail sales, focusing on the relationship between customer and pricing data, as well as validating pre-entered kpi data such as predicted_CLV and loyalty_score.

#### Key Objectives:

* **Category Mix Evolution: EDIT** 

---

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("RetailEDA") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

df = spark.read.csv(
    "../synthetic-datasets/datasets/retail/synthetic_retail_20250901.csv",
    header=True,
    inferSchema=True
)

df.printSchema()
df.show(5)

root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_type: string (nullable = true)
 |-- store_location: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- total_value: double (nullable = true)
 |-- customer_age: integer (nullable = true)
 |-- customer_income: double (nullable = true)
 |-- loyalty_score: double (nullable = true)
 |-- days_since_last_purchase: integer (nullable = true)
 |-- purchase_frequency_monthly: double (nullable = true)
 |-- average_order_value: double (nullable = true)
 |-- discount_percentage: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- sales_channel: string (nullable = true)
 |-- season: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- customer_registration_date: date (nullable = true)
 |-- customer_segment: integer (nullable = true)
 |-- pre

### 📊 Strategic Customer Segmentation & Pricing Analysis

1. **CLV Validation:** Made a validation analysis of the predicted_clv column to see if it can be used for future modeling.
    - 1.1. **customer_segment:** It was found this column has no predictive or discriminatory power for revenue behavior since the top spenders are scattared across all segments in almost exact proportions and should be re-evaluated in Gold layer
    - 1.2. **predicted_clv & loyalty_score:** It was found these columns have almost no relationship with how the customers actually behave -- almost zero correlation with total_value, almost zero correlation with purchase_frequency_monthly, barely any correlation with avg_order_value -- should be re-evaluated in Gold layer
2. **Tenure Analysis:** Identify patterns in transaction_count and average_spending by customer loyalty age.
    - 2.1. **first_registration_date:** The previous EDA, it was found that all customer_ids have multiple registration_dates, so a canonical one was created as baseline for tenure
    - 2.2. **Flat Line:** It was found through Tenure that I cannot rely on customer value to grow my CLV, and retention starts to slowly fall off near the last transaction_dates in the dataset. Other analysis are necessary to understand what does make frequency increase and recover retention over time

In [2]:
#   ═════════════════════════════════════════════════════════════════════════════
#   CLV Validation
#   Is predicted_clv consistent with observed transaction behavior?
#   ═════════════════════════════════════════════════════════════════════════════

# Check if high-value segments (e.g. = 0) are actually spending more

customer_segment_analysis = df.groupBy("customer_segment").agg(
    F.avg("total_value").alias("avg_transaction_value"),
    F.sum("total_value").alias("total_revenue_contribution"), # See who brings in the most cash
    F.avg("purchase_frequency_monthly").alias("avg_frequency"), # Are some segments buying more often?
    F.count("*").alias("segment_size")
).orderBy(F.desc("total_revenue_contribution"))

customer_segment_analysis.show()

+----------------+---------------------+--------------------------+------------------+------------+
|customer_segment|avg_transaction_value|total_revenue_contribution|     avg_frequency|segment_size|
+----------------+---------------------+--------------------------+------------------+------------+
|               0|    66.35505774605788|       3.030170067031479E7|15.101690951808425|      456660|
|               1|     66.4358541427228|      2.0681414958775464E7|15.519964527644811|      311299|
|               2|    64.52573981244542|        2043981.8600388337|15.644557606619987|       31677|
|               3|    68.33038525216665|         24872.26023178866| 18.96694214876033|         364|
+----------------+---------------------+--------------------------+------------------+------------+



In [3]:
# Identify the top 1% spending regardless of segment and see which segments we can find "Whales" in

top_threshold = df.select(F.percentile_approx("total_value", 0.99)).collect()[0][0]
print(f"Top 1% Spend Threshold: {top_threshold}")

df.filter(F.col("total_value") > top_threshold) \
  .groupBy("customer_segment") \
  .count() \
  .orderBy(F.desc("count")) \
  .show()

Top 1% Spend Threshold: 559.4930696431886
+----------------+-----+
|customer_segment|count|
+----------------+-----+
|               0| 4606|
|               1| 3204|
|               2|  266|
|               3|    4|
+----------------+-----+



In [5]:
# Check if predicted_clv & loyalty_score actually correlate with observed spending patterns

df.select(
    corr("predicted_clv", "total_value").alias("clv_vs_total_value"),
    corr("predicted_clv", "purchase_frequency_monthly").alias("clv_vs_frequency"),
    corr("predicted_clv", "average_order_value").alias("clv_vs_avg_order_value")
).show()

df.select(
    corr("loyalty_score", "total_value").alias("loyalty_vs_total_value"),
    corr("loyalty_score", "purchase_frequency_monthly").alias("loyalty_vs_frequency"),
    corr("loyalty_score", "average_order_value").alias("loyalty_vs_avg_order_value")
).show()

+--------------------+--------------------+----------------------+
|  clv_vs_total_value|    clv_vs_frequency|clv_vs_avg_order_value|
+--------------------+--------------------+----------------------+
|-0.00195058747939...|0.001332813717515...|  0.024040213128206752|
+--------------------+--------------------+----------------------+

+----------------------+--------------------+--------------------------+
|loyalty_vs_total_value|loyalty_vs_frequency|loyalty_vs_avg_order_value|
+----------------------+--------------------+--------------------------+
|  -0.00202657938427...|-0.09761098851106238|      4.419299396125132E-4|
+----------------------+--------------------+--------------------------+



In [ ]:
#   ═════════════════════════════════════════════════════════════════════════════
#   Customer Tenure Analysis
#   ═════════════════════════════════════════════════════════════════════════════

# It was found in the first EDA that all customers have multiple registration dates,
# So first, i'll make a simple transformation by creating a canonical customer table (without touching the transactions) as reference for my analysis

canonical_df = df \
    .groupBy("customer_id") \
    .agg(F.min("customer_registration_date").alias("first_registration_date"))

print(f"Canonical customer count: {canonical_df.count()}")
print(f"Unique customers in raw: {df.select('customer_id').distinct().count()}")
print(f"Total transactions (raw): {df.count()}")

df_tenure = df \
    .drop("customer_registration_date") \
    .join(canonical_df, on="customer_id", how="left")

print(f"Total transactions (after join): {df_tenure.count()}")

In [ ]:
df_tenure_final = df_tenure.withColumn("tenure_month_start", 
    (F.floor(F.datediff(F.col("transaction_date"), F.col("first_registration_date")) / 30))
)

df_tenure_final = df_tenure_final.withColumn("tenure_bucket", 
    F.concat(F.col("tenure_month_start").cast("string"), F.lit("-"), 
             (F.col("tenure_month_start") + 1).cast("string"), F.lit(" Months"))
)

tenure_analysis = df_tenure_final.groupBy("tenure_month_start", "tenure_bucket").agg(
    F.count("*").alias("tx_count"),
    F.avg("total_value").alias("avg_basket"),
    F.sum("total_value").alias("total_rev")
).orderBy("tenure_month_start") # This fixes the Cast Error

tenure_analysis.select("tenure_bucket", "tx_count", "avg_basket", "total_rev").show(40, truncate=False)

In [ ]:
df_tenure_final.select("tenure_month_start").show()

### Log:

Transformation rules discovered here will be implemented in Silver:
- predicted_clv & loyalty_score is wrong
- customer_segment is wrong
- customer_registration_date is wrong